In [ ]:
import sys
sys.path.append('/Workspace/Repos/filipshahanski@gmail.com/dbx-ingestion-framework/modules')
from connections import SFTPClient

In [ ]:
configs = spark.table("config.sftp_config").filter("enabled = true").collect()

for config in configs:

    password = dbutils.secrets.get(
        scope=config.secret_scope,
        key=config.secret_key
    )

    with SFTPClient(
        host=config.host,
        port=config.port,
        username=config.username,
        password=password,
    ) as client:

        downloaded_files = client.retrieve_multiple_files(
            remote_path=config.remote_path,
            filename_pattern=config.filename_pattern,
            local_path=config.volume_path,
        )

        if config.archive_action == "ARCHIVE":
            client.archive_files(
                remote_path=config.remote_path,
                archive_path=config.archive_path,
                filenames=downloaded_files,
            )

        elif config.archive_action == "DELETE":
            client.delete_files(
                remote_path=config.remote_path,
                filenames=downloaded_files,
            )